In [ ]:
from nba_api.stats import endpoints
import pandas as pd

response = endpoints.SynergyPlayTypes(play_type_nullable='Transition', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
transition_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Isolation', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
isolation_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='PRBallHandler', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
PNRBH_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='PRRollman', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
PNRRM_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='OffRebound', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Putback_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Spotup', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Spotup_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Cut', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Cut_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Handoff', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Handoff_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='OffScreen', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
OffScreen_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Misc', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Misc_df = response.get_data_frames()[0]
response = endpoints.SynergyPlayTypes(play_type_nullable='Postup', player_or_team_abbreviation='P', type_grouping_nullable='Offensive')
Postup_df = response.get_data_frames()[0]
dfs = [transition_df, isolation_df, PNRBH_df, PNRRM_df, Putback_df, Spotup_df, Cut_df, Handoff_df, OffScreen_df, Misc_df, Postup_df]
combined_df = pd.concat(dfs, ignore_index=True)
pivot_df = combined_df.pivot_table(index='PLAYER_NAME',  # Player name as index
                                   columns='PLAY_TYPE',  # Play types as separate columns
                                   values='PTS',  # Fill with points per game
                                   aggfunc='first'  # Assuming you want to keep the first encountered value if there's any duplicate
                                  ).reset_index()
player_team_df = combined_df.groupby('PLAYER_NAME')['TEAM_ABBREVIATION'].agg('first').reset_index()

# Step 2: Merge the team information with pivot_df
pivot_df = pd.merge(pivot_df, player_team_df, on='PLAYER_NAME', how='left')

# The pivot_df now has one row per player with each play type's points per game as separate columns.
sums = pivot_df.drop(['PLAYER_NAME','TEAM_ABBREVIATION'], axis=1).sum(axis=1)

pivot_df['Sum'] = sums
playtypes = ['Cut', 'Isolation', 'PRRollMan', 'PRBallHandler', 'OffRebound', 'Spotup', 'Handoff', 'OffScreen', 'Misc', 'Postup','Transition']

for playtype in playtypes:
    percentage_column_name = playtype + '%' 
    pivot_df[percentage_column_name] = pivot_df[playtype] / pivot_df['Sum'] * 100  
pivot_df.drop(playtypes, axis=1, inplace=True)
pivot_df.drop('Sum',axis = 1, inplace=True)
pivot_df.fillna(0, inplace=True)
pivot_df

In [ ]:
cands_df = endpoints.LeagueDashPlayerPtShot(general_range_nullable = 'Catch and Shoot').get_data_frames()[0]
pullups_df = endpoints.LeagueDashPlayerPtShot(general_range_nullable = 'Pullups').get_data_frames()[0]
within10_df = endpoints.LeagueDashPlayerPtShot(general_range_nullable = 'Less Than 10 ft').get_data_frames()[0]

shooting_df = pd.merge(cands_df,pullups_df, on = "PLAYER_NAME", how = 'inner')
shooting_df = pd.merge(shooting_df, within10_df, on = 'PLAYER_NAME', how = 'inner')
shooting_df = shooting_df[[col for col in shooting_df.columns if 'FGA_FREQUENCY' in col or 'PLAYER_NAME' in col]]
shooting_df = shooting_df.rename(columns={"FGA_FREQUENCY_x": "FREQ_CANDS", "FGA_FREQUENCY_y": "FREQ_PULLUP", "FGA_FREQUENCY": "FREQ_WITHIN10"})

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from kneed import KneeLocator

player_df = pivot_df.copy()
scoring_df = endpoints.LeagueDashPlayerStats(measure_type_detailed_defense='Scoring').get_data_frames()[0]
scoring_df = scoring_df[scoring_df["GP"] > 20]
scoring_df = scoring_df[scoring_df["MIN"] > 15]
usage_df = endpoints.LeagueDashPlayerStats(measure_type_detailed_defense='Usage').get_data_frames()[0]
usage_df['MIN/G'] = usage_df['MIN'] / usage_df['GP']
usage_df = usage_df[usage_df["GP"] > 20]
usage_df = usage_df[usage_df['MIN/G'] > 15]
usage_df = usage_df[['PLAYER_NAME', 'USG_PCT']]
remove = [col for col in scoring_df.columns if 'PCT' not in col and col != 'PLAYER_NAME']
scoring_df.drop(columns  = remove, inplace = True, axis = 1)

zone_df = endpoints.LeagueDashPlayerShotLocations(distance_range = 'By Zone').get_data_frames()[0]
new_columns = [f'{col[0]}_{col[1]}' if col[0] else col[1] for col in zone_df.columns]

new_columns = [col.strip('_') for col in new_columns]
 
zone_df.columns = new_columns

removed_columns = [col for col in new_columns if 'Backcourt' in col or ('FGA' not in col and 'PLAYER_NAME' not in col) or 'Left' in col or 'Right' in col]

zone_df.drop(columns = removed_columns, inplace = True, axis = 1)

zone_df['Sum'] = zone_df.drop(['PLAYER_NAME'], axis=1).sum(axis=1)

columns  = [col for col in zone_df.columns if 'FGA' in col]
for column in columns:
    zone_df[column] = zone_df[column] / zone_df['Sum']
    
del zone_df['Sum']

bio_df = endpoints.LeagueDashPlayerBioStats().get_data_frames()[0]
bio_df = bio_df[['PLAYER_NAME', 'PLAYER_HEIGHT_INCHES']]

drives_df = pd.read_excel(r"C:\Users\chris\OneDrive\Documents\drives_nba.xlsx")
#drives_df = pd.read_excel(r"C:\Users\chris\OneDrive\Documents\drives_nba.xlsx")
drives_df['Drives/Min'] = drives_df['DRIVES'] / drives_df['MIN']
drives_df['PLAYER_NAME'] = drives_df['PLAYER']
drives_df = drives_df[['PLAYER_NAME','Drives/Min']]

#merged_df = pd.merge(pivot_df, zone_df, on='PLAYER_NAME', how='left')
#merged_df = pd.merge(merged_df,scoring_df, on= 'PLAYER_NAME', how = 'inner')
merged_df = pd.merge(pivot_df,usage_df, on= 'PLAYER_NAME', how = 'left')
merged_df = pd.merge(merged_df,bio_df, on= 'PLAYER_NAME', how = 'left')
#merged_df = pd.merge(merged_df,drives_df, on= 'PLAYER_NAME', how = 'left')
merged_df = pd.merge(merged_df, shooting_df, on = 'PLAYER_NAME', how = 'left')
columns  = [col for col in merged_df.columns if 'RANK' in col or 'W_PCT' in col or 'TEAM_ABBREVIATION' in col]
merged_df.drop(columns = columns, inplace = True, axis = 1)
#del merged_df['PCT_UAST_3PM']
#del merged_df['PCT_AST_3PM']
del merged_df['Misc%']
#del merged_df['FG_PCT']

# Prepare the data
X = merged_df.drop('PLAYER_NAME', axis=1)
feature_names = X.columns

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Choose number of components based on explained variance
n_components = np.argmax(np.cumsum(pca.explained_variance_ratio_) > 0.90) + 1

# Refit PCA with chosen number of components
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

silhouette_scores = []
inertias = []  # For elbow method
K = range(2, 15)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_pca)
    silhouette_scores.append(silhouette_score(X_pca, kmeans.labels_))
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(K, silhouette_scores, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. Number of Clusters')

# Silhouette score method
silhouette_optimal_k = K[np.argmax(silhouette_scores)]
plt.axvline(x=silhouette_optimal_k, color='r', linestyle='--', label=f'Optimal k={silhouette_optimal_k}')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(K, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')

# Elbow method
kn = KneeLocator(K, inertias, curve='convex', direction='decreasing')
elbow_optimal_k = kn.knee
plt.axvline(x=elbow_optimal_k, color='r', linestyle='--', label=f'Optimal k={elbow_optimal_k}')
plt.legend()

plt.tight_layout()
plt.show()


# Perform k-means with optimal k
optimal_k = 7
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
cluster_labels = kmeans.fit_predict(X_pca)

merged_df['Cluster'] = cluster_labels

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis')
plt.title('K-means Clustering of NBA Players (PCA)')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.colorbar(scatter)
plt.show()

# Analyze clusters
for i in range(optimal_k):
    print(f"\nCluster {i}:")
    print(merged_df[merged_df['Cluster'] == i]['PLAYER_NAME'].tolist())

# Identify most important features for each principal component
component_df = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(n_components)], index=feature_names)
for i in range(n_components):
    print(f"\nTop 5 features for PC{i+1}:")
    print(component_df[f'PC{i+1}'].abs().sort_values(ascending=False).head())

In [ ]:
cluster_centers = kmeans.cluster_centers_
cluster_centers_df = pd.DataFrame(scaler.inverse_transform(pca.inverse_transform(cluster_centers)), 
                                  columns=feature_names)
pd.set_option('display.max_columns', None)
cluster_centers_df

In [ ]:
"""
Cluster 0: 3 & D wings who run in transition"
Cluster 1: Traditional bigs that mainly score around the basket"
Cluster 2: Primary ball-handlers that run the offense and generate their own shots
Cluster 3: All around players that finish plays but dont't dominate the ball
Cluster 4: Catch and Shoot floor spacers that don't really do much else
Cluster 5: 3 point specialists that teams run plays for
Cluster 6: Do it all bigs that post up and generate offense for themselves
Cluster 7: Secondary ball-handlers that play off ball too
Cluster 8: Versatile bigs that can both play inside and outside
"""

In [ ]:
usage_df = endpoints.LeagueDashPlayerStats(measure_type_detailed_defense='Usage').get_data_frames()[0]

In [ ]:
usage_df